<h1> Model Development </h1>

This notebook develops predictive models for identifying hepatic steatosis using the NHANES 2017–March 2020 training cohort produced in `02-metabolic-phenotyping.ipynb`. The held-out test cohort is not accessed throughout model development so that it remains independent of preprocessing, fitting and hyperparameter selection, allowing an unbiased final assessment of model performance on unseen participants in `04-evaluation-interpretation.ipynb`.

Three models are developed:

1. Logistic regression as an interpretable baseline
2. XGBoost using demographic, clinical and metabolic predictors
3. XGBoost using the same predictors with metabolic phenotype added

XGBoost hyperparameters are selected using five-fold stratified cross-validation on the training cohort, with mean validation AUROC used as the selection criterion. The selected hyperparameters are used for both XGBoost models so that phenotype membership remains the only deliberate difference between them.

All preprocessing and model fitting are performed using training data only. The final fitted pipelines and XGBoost tuning results are saved for subsequent evaluation and interpretation.

<h2> Importing libraries and defining paths </h2>

The required libraries for data handling, preprocessing, pipeline construction, cross-validation, hyperparameter tuning and model storage are imported.

A fixed random state of `42` is used to support reproducibility. `CV_FOLDS = 5` specifies five-fold cross-validation, while `N_ITER = 100` specifies that 100 randomly selected XGBoost hyperparameter combinations will be evaluated.

Paths are defined for loading the training cohort from `../data/processed`, saving the fitted model pipelines and selected-parameter summary under `../data/models`, and saving the complete hyperparameter-tuning results under `../results`. The required output directories are created if they do not already exist.

In [1]:
from pathlib import Path
import json

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

RANDOM_STATE = 42
CV_FOLDS = 5
N_ITER = 100

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = DATA_DIR / "models"
RESULTS_DIR = Path("../results")

TRAIN_DATA_PATH = PROCESSED_DIR / "train_clustered.csv"

CV_RESULTS_PATH = RESULTS_DIR / "xgb_tuning_results.csv"
BEST_PARAMS_PATH = MODEL_DIR / "xgb_best_parameters.json"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

<h2> Loading and validating the training cohort </h2>

The training cohort produced by `02-metabolic-phenotyping.ipynb` is loaded from `train_clustered.csv`. The dataset contains the selected demographic, clinical and metabolic predictors, hepatic steatosis outcome, numerical cluster assignment and descriptive metabolic phenotype for each training participant.

The number of participants and variables, hepatic steatosis outcome distribution and metabolic phenotype distribution are displayed. The first five training records are also displayed to verify the dataset's structure and contents.

In [2]:
train_df = pd.read_csv(TRAIN_DATA_PATH)

required_columns = [
    "SEQN",
    "age",
    "sex",
    "race_ethnicity",
    "bmi",
    "waist_cm",
    "systolic_bp",
    "diastolic_bp",
    "hdl_mg_dl",
    "triglycerides_mg_dl",
    "glucose_mg_dl",
    "diabetes_status",
    "cap_db_m",
    "hepatic_steatosis",
    "cluster",
    "metabolic_phenotype",
]

missing_columns = [
    column for column in required_columns
    if column not in train_df.columns
]

expected_phenotypes = {
    "Lower metabolic-risk",
    "Higher metabolic-risk",
}

assert not missing_columns, f"Missing columns: {missing_columns}"
assert train_df["SEQN"].is_unique
assert train_df[required_columns].isna().sum().sum() == 0
assert set(train_df["hepatic_steatosis"].unique()) == {0, 1}
assert set(train_df["cluster"].unique()) == {0, 1}
assert (
    set(train_df["metabolic_phenotype"].unique())
    == expected_phenotypes
)

print(f"Training participants: {len(train_df):,}")
print(f"Variables:             {train_df.shape[1]}")

print("\nTraining outcome distribution:")
print(
    train_df["hepatic_steatosis"]
    .value_counts()
    .sort_index()
)

print("\nTraining metabolic phenotypes:")
print(train_df["metabolic_phenotype"].value_counts())

print("\nFirst five training records:")
display(train_df.head())

Training participants: 2,616
Variables:             16

Training outcome distribution:
hepatic_steatosis
0    1110
1    1506
Name: count, dtype: int64

Training metabolic phenotypes:
metabolic_phenotype
Lower metabolic-risk     1463
Higher metabolic-risk    1153
Name: count, dtype: int64

First five training records:


,SEQN,age,sex,race_ethnicity,bmi,waist_cm,systolic_bp,diastolic_bp,hdl_mg_dl,triglycerides_mg_dl,glucose_mg_dl,diabetes_status,cap_db_m,hepatic_steatosis,cluster,metabolic_phenotype
0,110604.0,19.0,Male,Non-Hispanic White,21.9,79.0,99.666667,66.333333,41.0,55.0,97.0,No,228.0,0,0,Lower metabolic-risk
1,117697.0,19.0,Female,Other Hispanic,24.5,78.6,104.666667,67.666667,62.0,48.0,87.0,No,211.0,0,0,Lower metabolic-risk
2,124665.0,65.0,Male,Non-Hispanic Black,33.7,116.4,143.666667,82.666667,38.0,133.0,128.0,No,373.0,1,1,Higher metabolic-risk
3,111297.0,70.0,Male,Non-Hispanic White,32.4,116.4,120.666667,73.333333,38.0,156.0,107.0,No,268.0,1,1,Higher metabolic-risk
4,121483.0,18.0,Female,Non-Hispanic White,23.3,81.8,106.333333,65.666667,51.0,49.0,104.0,No,196.0,0,0,Lower metabolic-risk


The validation checks confirm that the training dataset contains all expected columns, unique participant identifiers (`SEQN`), no missing values in the required variables, both hepatic steatosis outcome classes, both cluster labels and both metabolic phenotype categories. The training cohort contains 2,616 participants.

<h2> Defining the outcome and predictor sets </h2>

The binary outcome is `hepatic_steatosis`, where `1` and `0` represent the presence and absence of hepatic steatosis, respectively.

The `base_features` set contains eight numerical and three categorical variables representing demographic, anthropometric, clinical and metabolic information.

A second predictor set, `features_with_phenotype`, adds `metabolic_phenotype`, a binary categorical variable indicating membership in the lower or higher metabolic-risk phenotype identified through K-means clustering of the training cohort in `02-metabolic-phenotyping.ipynb`. CAP measurements and the `hepatic_steatosis` outcome were not used to derive these phenotypes.

`base_features` is used for logistic regression and XGBoost without phenotype information, while `features_with_phenotype` is used for XGBoost with phenotype information. This allows the two final XGBoost models to use the same selected hyperparameters while differing only in whether phenotype membership is included as a predictor.

`SEQN` is excluded because it is a participant identifier, while `cap_db_m` is excluded to prevent outcome leakage because the hepatic steatosis outcome is derived directly from CAP. The numerical `cluster` variable is also excluded because it contains the same phenotype membership information as `metabolic_phenotype`.

In [3]:
outcome = "hepatic_steatosis"

numeric_features = [
    "age",
    "bmi",
    "waist_cm",
    "systolic_bp",
    "diastolic_bp",
    "hdl_mg_dl",
    "triglycerides_mg_dl",
    "glucose_mg_dl",
]

categorical_features = [
    "sex",
    "race_ethnicity",
    "diabetes_status",
]

base_features = numeric_features + categorical_features

phenotype_feature = ["metabolic_phenotype"]
features_with_phenotype = base_features + phenotype_feature

print(f"Base predictors:           {len(base_features)}")
print(f"Predictors with phenotype: {len(features_with_phenotype)}")

print("\nBase predictors:")
for feature in base_features:
    print(f"- {feature}")

print("\nAdditional phenotype predictor:")
print(f"- {phenotype_feature[0]}")

Base predictors:           11
Predictors with phenotype: 12

Base predictors:
- age
- bmi
- waist_cm
- systolic_bp
- diastolic_bp
- hdl_mg_dl
- triglycerides_mg_dl
- glucose_mg_dl
- sex
- race_ethnicity
- diabetes_status

Additional phenotype predictor:
- metabolic_phenotype


<h2> Separating predictors and outcome </h2>

The training cohort is separated into predictor datasets (`X`) and the hepatic steatosis outcome variable (`y_train`).

Two versions of the training predictors are prepared. `X_train_base` contains the 11 demographic, clinical and metabolic predictors, while `X_train_with_phenotype` additionally contains the `metabolic_phenotype` predictor.

The dimensions of both predictor datasets and the outcome variable are displayed.

In [4]:
# Base training predictors
X_train_base = train_df[base_features].copy()

# Training predictors including metabolic phenotype
X_train_with_phenotype = train_df[
    features_with_phenotype
].copy()

# Training outcome
y_train = train_df[outcome].copy()

# Validate predictor-outcome alignment
assert X_train_base.index.equals(y_train.index)
assert X_train_with_phenotype.index.equals(y_train.index)

print(f"X_train_base:             {X_train_base.shape}")
print(f"X_train_with_phenotype:   {X_train_with_phenotype.shape}")
print(f"y_train:                  {y_train.shape}")

X_train_base:             (2616, 11)
X_train_with_phenotype:   (2616, 12)
y_train:                  (2616,)


The validation checks confirm that the rows in both training predictor datasets remain aligned with their corresponding outcome values. The base predictor dataset contains 2,616 rows and 11 predictors, while the phenotype-enhanced dataset contains the same 2,616 rows and 12 predictors.

<h2> Defining preprocessing pipelines </h2>

For logistic regression, numerical predictors are standardised using `StandardScaler` because its coefficients are sensitive to differences in predictor scale. Categorical predictors are converted into numerical indicator variables using `OneHotEncoder`. The `Female`, `Non-Hispanic White` and `No` categories are explicitly omitted as the reference categories for `sex`, `race_ethnicity` and `diabetes_status`, respectively.

For XGBoost, numerical predictors are retained in their original units because tree-based models do not require standardisation. Hence, no numerical preprocessing parameters are learned. Categorical predictors are one-hot encoded, with all categories retained. The encoder learns the categories and resulting indicator columns from the relevant training data. The phenotype-enhanced preprocessor additionally includes `metabolic_phenotype` as a categorical predictor.

At this stage, the preprocessing objects are defined but not yet fitted. For logistic regression and the final XGBoost models, preprocessing is fitted using the complete training cohort as part of each model pipeline. During XGBoost hyperparameter tuning, the base preprocessor is fitted separately within each cross-validation training fold and then applied to its corresponding validation fold without refitting. This prevents information from the validation fold from influencing preprocessing.

`handle_unknown="ignore"` is specified as a safeguard so that categories absent during model fitting but present in subsequent data do not cause an encoding error.

In [5]:
# Logistic regression:
# standardise numerical predictors and one-hot encode categorical predictors
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=[
                    "Female",
                    "Non-Hispanic White",
                    "No",
                ],
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
)

# XGBoost without phenotype:
# retain numerical values and one-hot encode categorical predictors
xgb_base_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ],
    remainder="drop",
)

# XGBoost with phenotype:
# add metabolic phenotype as another categorical predictor
xgb_phenotype_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features + phenotype_feature,
        ),
    ],
    remainder="drop",
)

print("Preprocessing pipelines defined successfully.")

Preprocessing pipelines defined successfully.


<h2> Developing the predictive models </h2>

<h3> Logistic regression baseline </h3>

A pipeline combines the logistic-regression preprocessor with a multivariable logistic regression classifier. When the pipeline is fitted, the preprocessor first learns and applies the required transformations to the training predictors. The classifier then uses the transformed predictors together with the corresponding training outcomes to estimate its coefficients.

Logistic regression provides a comparatively simple and interpretable baseline for comparison with the XGBoost models. The maximum number of iterations is set to `2,000` to allow the optimisation process sufficient opportunity to converge.

The complete pipeline is fitted using all 2,616 training participants. Test probabilities and classifications are not generated during model development.

In [6]:
logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            logistic_preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

# Preprocessor learns transformation rules, classifier learns predictive relationship from transformed training data
logistic_model.fit(X_train_base, y_train) 

fitted_logistic_classifier = (
    logistic_model.named_steps["classifier"]
)

assert set(fitted_logistic_classifier.classes_) == {0, 1}

print("Logistic regression fitted successfully.")
print(
    "Outcome classes: "
    f"{fitted_logistic_classifier.classes_.tolist()}"
)

Logistic regression fitted successfully.
Outcome classes: [0, 1]


The logistic regression pipeline was fitted successfully, and the fitted classifier contains both hepatic steatosis outcome classes (`0` and `1`). The absence of a convergence warning indicates that model fitting completed normally.

The fitted pipeline is retained for subsequent saving and held-out evaluation in `04-evaluation-interpretation.ipynb`.

<h3> XGBoost hyperparameter tuning </h3>

An XGBoost pipeline is constructed using the base preprocessor and an `XGBClassifier`. Hyperparameter tuning is performed using only the 2,616-participant training cohort; the held-out test cohort is not accessed.

Six commonly tuned hyperparameters are varied because they control complementary aspects of XGBoost:

- `n_estimators` and `learning_rate` govern the number and contribution of boosted trees
- `max_depth` and `min_child_weight` regulate tree complexity
- `subsample` and `colsample_bytree` control the proportions of training participants and transformed predictor columns available when constructing each tree

The candidate ranges were selected to explore moderate variations in these properties while balancing model flexibility, overfitting risk and computational cost. The candidate values form 3 × 3 × 4 × 3 × 3 × 3 = 972 possible hyperparameter combinations. Exhaustively evaluating every combination using five-fold cross-validation would require 4,860 cross-validation fits. To limit computation, `RandomizedSearchCV` randomly and reproducibly samples 100 combinations (`N_ITER = 100`), resulting in 500 cross-validation fits.

Five-fold stratified cross-validation is used so that each candidate configuration is trained on four folds and validated on the remaining fold, with each fold serving as the validation fold once. Stratification preserves approximately the same hepatic steatosis outcome proportion in every fold.

Mean validation AUROC across the five folds is used to select the best configuration. Average precision is recorded as a secondary metric but is not used for parameter selection. The complete preprocessing-and-classifier pipeline is fitted separately within each fold, preventing information from its validation fold from influencing preprocessing or model fitting. After tuning, the configuration with the highest mean validation AUROC is refitted on the complete training cohort.

In [7]:
# Define the XGBoost pipeline used for hyperparameter tuning
xgb_tuning_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            xgb_base_preprocessor,
        ),
        (
            "classifier",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
        ),
    ]
)

# Candidate hyperparameter values
parameter_distributions = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__learning_rate": [0.01, 0.05, 0.10],
    "classifier__max_depth": [2, 3, 4, 5],
    "classifier__min_child_weight": [1, 3, 5],
    "classifier__subsample": [0.7, 0.8, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 1.0],
}

# Stratified cross-validation preserves outcome proportions in each fold
cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Search for the settings with the highest mean validation AUROC
xgb_search = RandomizedSearchCV(
    estimator=xgb_tuning_pipeline,
    param_distributions=parameter_distributions,
    n_iter=N_ITER,
    scoring={
        "auroc": "roc_auc",
        "average_precision": "average_precision",
    },
    refit="auroc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)

xgb_search.fit(
    X_train_base,
    y_train,
)

print("XGBoost hyperparameter tuning completed.")
print(f"Best mean cross-validated AUROC: {xgb_search.best_score_:.3f}")

print("\nBest parameters:")
for parameter, value in xgb_search.best_params_.items():
    print(f"- {parameter}: {value}")

Fitting 5 folds for each of 100 candidates, totalling 500 fits
XGBoost hyperparameter tuning completed.
Best mean cross-validated AUROC: 0.841

Best parameters:
- classifier__subsample: 0.7
- classifier__n_estimators: 500
- classifier__min_child_weight: 1
- classifier__max_depth: 3
- classifier__learning_rate: 0.01
- classifier__colsample_bytree: 0.8


The 100-candidate search completed successfully. The selected hyperparameter configuration achieved the highest mean five-fold cross-validated AUROC of `0.841149`.

The selected configuration uses 500 trees, a learning rate of `0.01`, a maximum tree depth of `3` and a minimum child weight of `1`. Each boosting iteration uses a random 70% subsample of the training participants, while each tree uses a random 80% subset of the transformed predictor columns.

These results describe cross-validation performance within the training cohort and do not represent final performance on the held-out test cohort.

<h3> Retaining the selected XGBoost model and tuning results </h3>

Following hyperparameter tuning, `best_estimator_` provides the complete base XGBoost pipeline refitted on the full training cohort using the selected hyperparameters. This fitted pipeline is retained as `xgb_base_model`.

The selected hyperparameters are extracted without their pipeline prefixes so that the same configuration can subsequently be used for the phenotype-enhanced XGBoost model. The complete tuning results are saved as `xgb_tuning_results.csv` to provide a tabular record of every evaluated configuration and its cross-validation performance. A concise summary is saved as `xgb_best_parameters.json` to preserve the search settings, selection metric, best mean cross-validated AUROC score and selected parameter-value pairs in a structured form. These files support reproducibility and allow the tuning process to be reviewed without rerunning the search. 

The five highest-ranked candidate configurations are displayed according to mean validation AUROC.

In [8]:
# Retain the best pipeline, already refitted on the full training cohort
xgb_base_model = xgb_search.best_estimator_

# Remove pipeline prefixes for reuse in another XGBClassifier
best_xgb_parameters = {
    parameter.replace("classifier__", ""): value
    for parameter, value in xgb_search.best_params_.items()
}

# Save complete cross-validation results
cv_results_df = (
    pd.DataFrame(xgb_search.cv_results_)
    .sort_values("rank_test_auroc")
    .reset_index(drop=True)
)

cv_results_df.to_csv(
    CV_RESULTS_PATH,
    index=False,
)

# Save a concise tuning summary
tuning_summary = {
    "selection_metric": "roc_auc",
    "cv_folds": CV_FOLDS,
    "candidates_tested": N_ITER,
    "best_mean_cv_auroc": float(xgb_search.best_score_),
    "best_parameters": best_xgb_parameters,
}

with open(BEST_PARAMS_PATH, "w") as file:
    json.dump(
        tuning_summary,
        file,
        indent=4,
    )

print("Best XGBoost pipeline retained and tuning results saved.")
print(f"Cross-validation results: {CV_RESULTS_PATH}")
print(f"Best-parameter summary:   {BEST_PARAMS_PATH}")

display(
    cv_results_df[
        [
            "rank_test_auroc",
            "mean_train_auroc",
            "mean_test_auroc",
            "std_test_auroc",
            "mean_test_average_precision",
            "params",
        ]
    ].head()
)

Best XGBoost pipeline retained and tuning results saved.
Cross-validation results: ../results/xgb_tuning_results.csv
Best-parameter summary:   ../data/models/xgb_best_parameters.json


,rank_test_auroc,mean_train_auroc,mean_test_auroc,std_test_auroc,mean_test_average_precision,params
0,1,0.886839,0.841149,0.020039,0.865918,"{'classifier__subsample': 0.7, 'classifier__n_..."
1,2,0.874261,0.840919,0.018852,0.864998,"{'classifier__subsample': 0.7, 'classifier__n_..."
2,3,0.875358,0.840581,0.018995,0.864878,"{'classifier__subsample': 0.7, 'classifier__n_..."
3,4,0.877792,0.840536,0.019646,0.866383,"{'classifier__subsample': 0.8, 'classifier__n_..."
4,5,0.885276,0.839856,0.019723,0.862429,"{'classifier__subsample': 0.7, 'classifier__n_..."


The selected base XGBoost pipeline was retained successfully. Results for all 100 evaluated configurations were saved under `../results`, while the selected-parameter summary was saved under `../data/models`.

The displayed table presents the five highest-ranked configurations, including their mean training AUROC, mean validation AUROC, variation in validation AUROC across folds and mean validation average precision.

<h3> Fitting the final XGBoost models </h3>

The base XGBoost pipeline selected by `RandomizedSearchCV` has already been refitted on the complete training cohort and retained as `xgb_base_model`.

A phenotype-enhanced pipeline is now constructed using the corresponding preprocessor and an `XGBClassifier`. It is fitted on the complete training cohort using the same selected hyperparameters as the base XGBoost model, with `metabolic_phenotype` included as the only additional predictor.

Reusing the selected configuration ensures that the inclusion of phenotype membership remains the only deliberate modelling difference between the two XGBoost pipelines. Both fitted classifiers are checked to confirm that they contain the two hepatic steatosis outcome classes.

In [9]:
xgb_phenotype_model = Pipeline(
    steps=[
        (
            "preprocessor",
            xgb_phenotype_preprocessor,
        ),
        (
            "classifier",
            XGBClassifier(
                **best_xgb_parameters,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
        ),
    ]
)

xgb_phenotype_model.fit(
    X_train_with_phenotype,
    y_train,
)

fitted_base_classifier = (
    xgb_base_model.named_steps["classifier"]
)

fitted_phenotype_classifier = (
    xgb_phenotype_model.named_steps["classifier"]
)

assert set(fitted_base_classifier.classes_) == {0, 1}
assert set(fitted_phenotype_classifier.classes_) == {0, 1}

print("Final XGBoost models fitted successfully.")
print("Both models use the selected training-derived hyperparameters.")

print("\nSelected hyperparameters:")
for parameter, value in best_xgb_parameters.items():
    print(f"- {parameter}: {value}")

Final XGBoost models fitted successfully.
Both models use the selected training-derived hyperparameters.

Selected hyperparameters:
- subsample: 0.7
- n_estimators: 500
- min_child_weight: 1
- max_depth: 3
- learning_rate: 0.01
- colsample_bytree: 0.8


Both final XGBoost pipelines were fitted successfully using the selected training-derived hyperparameters. The base model uses the 11 demographic, clinical and metabolic predictors, while the phenotype-enhanced model additionally includes `metabolic_phenotype`.

No held-out test predictions or performance comparisons are produced at this stage. Both fitted pipelines are retained for evaluation in `04-evaluation-interpretation.ipynb`.

<h2> Saving the fitted model pipelines </h2>

The logistic regression pipeline and both XGBoost pipelines are saved under `../data/models` using `joblib`. Each saved object contains the fitted preprocessing steps together with its fitted classifier.

Saving complete pipelines ensures that future data are processed using the same training-derived transformations before predictions are generated. It also allows `04-evaluation-interpretation.ipynb` to load and evaluate the fitted models without repeating preprocessing, hyperparameter tuning or model fitting.

In [10]:
model_paths = {
    "logistic_regression": (
        MODEL_DIR / "logistic_regression.joblib"
    ),
    "xgboost_without_phenotype": (
        MODEL_DIR / "xgboost_without_phenotype.joblib"
    ),
    "xgboost_with_phenotype": (
        MODEL_DIR / "xgboost_with_phenotype.joblib"
    ),
}

joblib.dump(
    logistic_model,
    model_paths["logistic_regression"],
)

joblib.dump(
    xgb_base_model,
    model_paths["xgboost_without_phenotype"],
)

joblib.dump(
    xgb_phenotype_model,
    model_paths["xgboost_with_phenotype"],
)

print("Saved fitted model pipelines:")
for name, path in model_paths.items():
    print(f"- {name}: {path}")

Saved fitted model pipelines:
- logistic_regression: ../data/models/logistic_regression.joblib
- xgboost_without_phenotype: ../data/models/xgboost_without_phenotype.joblib
- xgboost_with_phenotype: ../data/models/xgboost_with_phenotype.joblib


All three fitted model pipelines were saved successfully under `../data/models`. These frozen pipelines are subsequently loaded in `04-evaluation-interpretation.ipynb` to generate predictions for the held-out test cohort.

<h2> Validating the saved outputs </h2>

All fitted model, tuning-result and selected-parameter files are checked to confirm that they exist and are not empty. The three fitted models are reloaded and verified to be complete scikit-learn pipelines.

The saved cross-validation results are checked to confirm that they contain all 100 evaluated configurations, include a top-ranked configuration and contain valid AUROC and average-precision values between zero and one. The selected-parameter summary is reloaded and checked against the cross-validation design, selection metric and hyperparameters selected during tuning.

Finally, both saved XGBoost classifiers are checked to confirm that they use the same selected hyperparameter configuration.

In [11]:
# Confirm that all expected files exist and are not empty
output_paths = [
    *model_paths.values(),
    CV_RESULTS_PATH,
    BEST_PARAMS_PATH,
]

for path in output_paths:
    assert path.exists(), f"Missing output: {path}"
    assert path.stat().st_size > 0, f"Empty output: {path}"

# Reload and validate the fitted pipelines
loaded_models = {
    name: joblib.load(path)
    for name, path in model_paths.items()
}

assert all(
    isinstance(model, Pipeline)
    for model in loaded_models.values()
)

# Reload and validate the cross-validation results
saved_cv_results = pd.read_csv(CV_RESULTS_PATH)

assert len(saved_cv_results) == N_ITER
assert saved_cv_results["rank_test_auroc"].min() == 1
assert saved_cv_results["mean_test_auroc"].between(0, 1).all()
assert (
    saved_cv_results["mean_test_average_precision"]
    .between(0, 1)
    .all()
)

# Reload and validate the best-parameter summary
with open(BEST_PARAMS_PATH) as file:
    saved_tuning_summary = json.load(file)

assert saved_tuning_summary["selection_metric"] == "roc_auc"
assert saved_tuning_summary["cv_folds"] == CV_FOLDS
assert saved_tuning_summary["candidates_tested"] == N_ITER
assert saved_tuning_summary["best_parameters"] == best_xgb_parameters

# Confirm that both saved XGBoost models use the selected parameters
for model_name in [
    "xgboost_without_phenotype",
    "xgboost_with_phenotype",
]:
    classifier_parameters = (
        loaded_models[model_name]
        .named_steps["classifier"]
        .get_params()
    )

    for parameter, expected_value in best_xgb_parameters.items():
        assert classifier_parameters[parameter] == expected_value

print("Final validation checks passed.")
print(f"Saved fitted models:          {len(model_paths)}")
print(f"Saved tuning candidates:      {len(saved_cv_results)}")
print(f"Best cross-validated AUROC:   {xgb_search.best_score_:.6f}")
print(xgb_search.best_params_)

Final validation checks passed.
Saved fitted models:          3
Saved tuning candidates:      100
Best cross-validated AUROC:   0.841149
{'classifier__subsample': 0.7, 'classifier__n_estimators': 500, 'classifier__min_child_weight': 1, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.01, 'classifier__colsample_bytree': 0.8}


The final validation checks passed successfully. Three fitted model pipelines, results for all 100 evaluated hyperparameter configurations and the selected-parameter summary were saved and reloaded successfully.

The selected XGBoost configuration achieved a mean five-fold cross-validated AUROC of `0.841149`, with a standard deviation of approximately `0.020` across folds. This indicates promising discrimination within the training cohort, with relatively consistent performance across folds. However, this tuning-stage result only represents training-cohort cross-validation performance and does not establish whether the model performs adequately on unseen participants. Final predictive performance remains to be evaluated once using the held-out test cohort in `04-evaluation-interpretation.ipynb`.